In [1]:
import pandas as pd
df_train = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/FewRel/Train/MINT/FewRel_train_phrase_preprocess.csv')
print(df_train.shape) 
df_train.head(2)

(4000, 15)


,id,relation,relation_id,text,subject,object,subj_start,subj_end,obj_start,obj_end,subj_id,obj_id,token,gpt4_phrase,gpt4_phrase_preprocessed
0,train_P1877_557,after a work by,P1877,"After reading Alan Glynn 's "" The Dark Fields ...",limitless,alan glynn,14,14,2,3,Q19911394,Q2830507,"['After', 'reading', 'Alan', 'Glynn', ""'s"", '""...",inspired_by_author,inspired_by_author
1,train_P1877_158,after a work by,P1877,"It was a sequel to the 1952 film "" Heidi "" , w...",heidi,johanna spyri,9,9,25,26,Q455508,Q123053,"['It', 'was', 'a', 'sequel', 'to', 'the', '195...",adapted_from_author,adapted_from_author


In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize


texts = df_train["gpt4_phrase"].fillna("").astype(str).tolist()



print("num texts:", len(texts))


model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("embedding shape:", embeddings.shape)


def estimate_k_by_zhang2021(embeddings, k0, random_state=42):
    """
    Zhang et al. (2021a) 방식의 unknown K 추정.

    Args:
        embeddings   : clustering할 embedding matrix, shape = (N, dim)
        k0           : over-clustering에 사용할 큰 cluster 개수
        random_state : random seed

    Returns:
        estimated_k  : 추정된 cluster 개수
        labels_k0    : K0로 clustering한 결과
        cluster_info : cluster별 size 정보
    """

    n = len(embeddings)

    kmeans = KMeans(
        n_clusters=k0,
        random_state=random_state,
        n_init="auto"
    )

    labels_k0 = kmeans.fit_predict(embeddings)

    cluster_sizes = pd.Series(labels_k0).value_counts().sort_index()

    threshold = n / k0

    valid_clusters = cluster_sizes[cluster_sizes >= threshold].index.tolist()
    estimated_k = len(valid_clusters)

    cluster_info = pd.DataFrame({
        "cluster_id": cluster_sizes.index,
        "size": cluster_sizes.values,
        "threshold": threshold,
        "is_valid": cluster_sizes.values >= threshold
    })

    return estimated_k, labels_k0, cluster_info

num texts: 4000


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

embedding shape: (4000, 384)


In [3]:
k0 = 300

estimated_k, labels_k0, cluster_info = estimate_k_by_zhang2021(
    embeddings=embeddings,
    k0=k0,
    random_state=42
)

print("Estimated K:", estimated_k)
cluster_info

Estimated K: 96


,cluster_id,size,threshold,is_valid
0,0,15,13.333333,True
1,1,27,13.333333,True
2,2,214,13.333333,True
3,3,1,13.333333,False
4,4,18,13.333333,True
...,...,...,...,...
295,295,4,13.333333,False
296,296,1,13.333333,False
297,297,4,13.333333,False
298,298,11,13.333333,False


In [4]:
kmeans = KMeans(
    n_clusters=estimated_k,
    random_state=42,
    n_init="auto"
)

df_train["cluster_unknown_k"] = kmeans.fit_predict(embeddings)


# LLMEdgeRefine

In [5]:
import os
import re
import time
import numpy as np
import pandas as pd

from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import euclidean_distances


load_dotenv()

if "client" not in globals():
    api_key = os.getenv("OPENAI_API_KEY")
    client = OpenAI(api_key=api_key)


In [6]:


def relabel_to_consecutive(labels):
    labels = np.asarray(labels)
    unique_labels = sorted(np.unique(labels))
    mapping = {old_label: new_label for new_label, old_label in enumerate(unique_labels)}
    new_labels = np.array([mapping[x] for x in labels], dtype=int)
    return new_labels, mapping


def compute_centroids(embeddings, labels):
    labels = np.asarray(labels)
    unique_labels = sorted(np.unique(labels))

    centroids = []
    for c in unique_labels:
        idx = np.where(labels == c)[0]
        centroids.append(embeddings[idx].mean(axis=0))

    centroids = np.vstack(centroids)
    return centroids, unique_labels


def get_farthest_nodes_by_ratio(embeddings, labels, ratio):
    labels = np.asarray(labels)
    centroids, unique_labels = compute_centroids(embeddings, labels)

    selected_indices = []

    for local_c, cluster_id in enumerate(unique_labels):
        idx = np.where(labels == cluster_id)[0]

        if len(idx) == 0:
            continue

        cluster_emb = embeddings[idx]
        centroid = centroids[local_c].reshape(1, -1)
        dists = euclidean_distances(cluster_emb, centroid).reshape(-1)

        n_select = int(np.ceil(len(idx) * ratio))

        if n_select <= 0:
            continue

        farthest_local_idx = np.argsort(dists)[-n_select:]
        farthest_global_idx = idx[farthest_local_idx]

        selected_indices.extend(farthest_global_idx.tolist())

    selected_indices = sorted(list(set(selected_indices)))
    return selected_indices


def get_farthest_nodes_per_cluster(embeddings, labels, ratio):
    labels = np.asarray(labels)
    centroids, unique_labels = compute_centroids(embeddings, labels)

    result = {}

    for local_c, cluster_id in enumerate(unique_labels):
        idx = np.where(labels == cluster_id)[0]

        if len(idx) == 0:
            result[cluster_id] = []
            continue

        cluster_emb = embeddings[idx]
        centroid = centroids[local_c].reshape(1, -1)
        dists = euclidean_distances(cluster_emb, centroid).reshape(-1)

        n_select = int(np.ceil(len(idx) * ratio))

        if n_select <= 0:
            result[cluster_id] = []
            continue

        farthest_local_idx = np.argsort(dists)[-n_select:]
        farthest_global_idx = idx[farthest_local_idx].tolist()

        result[cluster_id] = farthest_global_idx

    return result


def build_cluster_description(df, embeddings, labels, cluster_id, text_col="gpt4_phrase", top_m=5):
    labels = np.asarray(labels)
    idx = np.where(labels == cluster_id)[0]

    if len(idx) == 0:
        return ""

    centroid = embeddings[idx].mean(axis=0).reshape(1, -1)
    dists = euclidean_distances(embeddings[idx], centroid).reshape(-1)

    nearest_local_idx = np.argsort(dists)[:min(top_m, len(idx))]
    nearest_global_idx = idx[nearest_local_idx]

    phrases = []
    for row_idx in nearest_global_idx:
        phrase = str(df.iloc[row_idx][text_col])
        phrases.append(phrase)

    description = "\n".join([f"- {p}" for p in phrases])
    return description


def super_point_enhanced_clustering(
    embeddings,
    init_labels,
    n_clusters,
    alpha=0.1,
    gamma=1
):
    labels = np.asarray(init_labels).copy()
    labels, _ = relabel_to_consecutive(labels)

    n = len(embeddings)

    for iteration in range(gamma):
        print(f"SPEC iteration {iteration + 1}/{gamma}")

        farthest_dict = get_farthest_nodes_per_cluster(
            embeddings=embeddings,
            labels=labels,
            ratio=alpha
        )

        used_indices = set()
        unit_embeddings = []
        unit_members = []

        for cluster_id, idx_list in farthest_dict.items():
            idx_list = sorted(list(set(idx_list)))

            if len(idx_list) == 0:
                continue

            used_indices.update(idx_list)

            super_point_embedding = embeddings[idx_list].mean(axis=0)
            unit_embeddings.append(super_point_embedding)
            unit_members.append(idx_list)

        for i in range(n):
            if i not in used_indices:
                unit_embeddings.append(embeddings[i])
                unit_members.append([i])

        unit_embeddings = np.vstack(unit_embeddings)

        agg = AgglomerativeClustering(
            n_clusters=n_clusters
        )

        unit_labels = agg.fit_predict(unit_embeddings)

        new_labels = np.zeros(n, dtype=int)

        for unit_idx, members in enumerate(unit_members):
            for original_idx in members:
                new_labels[original_idx] = unit_labels[unit_idx]

        labels, _ = relabel_to_consecutive(new_labels)

        print("num clusters:", len(np.unique(labels)))

    return labels


def build_llm_edge_refine_prompt(
    query_text,
    candidate_cluster_info,
    task_instruction=None
):
    if task_instruction is None:
        task_instruction = (
            "Select one classification of the relation phrase that better corresponds "
            "with the query in terms of semantic relation."
        )

    cluster_blocks = []

    for rank, (cluster_id, cluster_description) in enumerate(candidate_cluster_info, start=1):
        block = (
            f"Classification {rank}: Cluster {cluster_id}\n"
            f"Cluster description:\n{cluster_description}"
        )
        cluster_blocks.append(block)

    cluster_text = "\n\n".join(cluster_blocks)

    prompt = f"""
{task_instruction}

Query:
{query_text}

Classifications and cluster descriptions:
{cluster_text}

Return only the selected cluster id.
""".strip()

    return prompt


def parse_selected_cluster_id(output_text, valid_cluster_ids):
    text = str(output_text).strip()

    numbers = re.findall(r"-?\d+", text)

    for num in numbers:
        cid = int(num)
        if cid in valid_cluster_ids:
            return cid

    return None


def call_llm_select_cluster(
    client,
    prompt,
    valid_cluster_ids,
    model_name="gpt-4.1-mini",
    max_retries=3,
    sleep_sec=1.0
):
    last_error = None

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=model_name,
                input=prompt,
                temperature=0
            )

            output_text = response.output_text
            selected_cluster = parse_selected_cluster_id(
                output_text=output_text,
                valid_cluster_ids=valid_cluster_ids
            )

            if selected_cluster is not None:
                return selected_cluster, output_text, None

            last_error = f"Parsing failed. Raw output: {output_text}"

        except Exception as e:
            last_error = str(e)

        time.sleep(sleep_sec)

    return None, "", last_error


In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def llm_assisted_cluster_refinement(
    df,
    embeddings,
    init_labels,
    client,
    text_col="gpt4_phrase",
    beta=0.1,
    l_iter=3,
    n_demonstrations=8,
    desc_top_m=5,
    model_name="gpt-4.1-mini",
    task_instruction=None,
    log_save_path=None,
    num_workers=10
):
    labels = np.asarray(init_labels).copy()
    labels, _ = relabel_to_consecutive(labels)

    logs = []

    for iteration in range(l_iter):
        print(f"LACR iteration {iteration + 1}/{l_iter}")

        labels, _ = relabel_to_consecutive(labels)
        centroids, unique_labels = compute_centroids(embeddings, labels)
        k = len(unique_labels)

        edge_indices = get_farthest_nodes_by_ratio(
            embeddings=embeddings,
            labels=labels,
            ratio=beta
        )

        print("num edge points:", len(edge_indices))

        cluster_description_cache = {}

        for cluster_id in range(k):
            cluster_description_cache[cluster_id] = build_cluster_description(
                df=df,
                embeddings=embeddings,
                labels=labels,
                cluster_id=cluster_id,
                text_col=text_col,
                top_m=desc_top_m
            )

        new_labels = labels.copy()

        def process_one_edge_point(row_idx):
            query_text = str(df.iloc[row_idx][text_col])
            current_cluster = int(labels[row_idx])

            query_emb = embeddings[row_idx].reshape(1, -1)
            dists = euclidean_distances(query_emb, centroids).reshape(-1)

            nearest_cluster_ids = np.argsort(dists)[:min(n_demonstrations, k)].tolist()

            candidate_cluster_info = [
                (int(cluster_id), cluster_description_cache[int(cluster_id)])
                for cluster_id in nearest_cluster_ids
            ]

            prompt = build_llm_edge_refine_prompt(
                query_text=query_text,
                candidate_cluster_info=candidate_cluster_info,
                task_instruction=task_instruction
            )

            selected_cluster, raw_output, error_msg = call_llm_select_cluster(
                client=client,
                prompt=prompt,
                valid_cluster_ids=nearest_cluster_ids,
                model_name=model_name
            )

            if selected_cluster is None:
                should_remove = False
                reassigned_cluster = current_cluster
            else:
                should_remove = selected_cluster != current_cluster

                if should_remove:
                    reassigned_cluster = int(np.argmin(dists))
                else:
                    reassigned_cluster = current_cluster

            log_item = {
                "iteration": iteration + 1,
                "row_index": row_idx,
                "query_text": query_text,
                "current_cluster": current_cluster,
                "nearest_cluster_by_centroid": int(np.argmin(dists)),
                "candidate_clusters": nearest_cluster_ids,
                "llm_selected_cluster": selected_cluster,
                "should_remove": should_remove,
                "assigned_cluster_after_refinement": reassigned_cluster,
                "raw_output": raw_output,
                "error": error_msg
            }

            return row_idx, reassigned_cluster, log_item

        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = [
                executor.submit(process_one_edge_point, row_idx)
                for row_idx in edge_indices
            ]

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"LACR {iteration + 1}"
            ):
                row_idx, reassigned_cluster, log_item = future.result()
                new_labels[row_idx] = reassigned_cluster
                logs.append(log_item)

        labels = new_labels.copy()
        labels, _ = relabel_to_consecutive(labels)

        if log_save_path is not None:
            pd.DataFrame(logs).to_csv(log_save_path, index=False)
            print("saved log:", log_save_path)

        print("num clusters after LACR:", len(np.unique(labels)))

    log_df = pd.DataFrame(logs)
    return labels, log_df

In [8]:

init_labels = df_train["cluster_unknown_k"].to_numpy()
init_labels, _ = relabel_to_consecutive(init_labels)

n_clusters = len(np.unique(init_labels))
print("initial num clusters:", n_clusters)

alpha = 0.1
gamma = 1
beta = 0.1 
l_iter = 3  

labels_spec = super_point_enhanced_clustering(
    embeddings=embeddings,
    init_labels=init_labels,
    n_clusters=n_clusters,
    alpha=alpha,
    gamma=gamma
)

df_train["cluster_spec"] = labels_spec

task_instruction = (
    "Select one classification of the relation phrase that better corresponds "
    "with the query in terms of semantic relation."
)

labels_llmedgerefine, llmedgerefine_log_df = llm_assisted_cluster_refinement(
    df=df_train,
    embeddings=embeddings,
    init_labels=labels_spec,
    client=client,
    text_col="gpt4_phrase",
    beta=beta,
    l_iter=l_iter,
    n_demonstrations=8,
    desc_top_m=5,
    model_name="gpt-4.1-mini",
    task_instruction=task_instruction,
    log_save_path=None,
    num_workers=10
)

df_train["cluster_llmedgerefine"] = labels_llmedgerefine

print("cluster_unknown_k counts:")
print(df_train["cluster_unknown_k"].value_counts().sort_index())

print("cluster_spec counts:")
print(df_train["cluster_spec"].value_counts().sort_index())

print("cluster_llmedgerefine counts:")
print(df_train["cluster_llmedgerefine"].value_counts().sort_index())

llmedgerefine_log_df.head()

initial num clusters: 96
SPEC iteration 1/1
num clusters: 96
LACR iteration 1/3
num edge points: 443


LACR 1: 100%|██████████| 443/443 [01:02<00:00,  7.06it/s]


num clusters after LACR: 96
LACR iteration 2/3
num edge points: 442


LACR 2: 100%|██████████| 442/442 [01:11<00:00,  6.19it/s]


num clusters after LACR: 96
LACR iteration 3/3
num edge points: 439


LACR 3: 100%|██████████| 439/439 [01:07<00:00,  6.51it/s]

num clusters after LACR: 96
cluster_unknown_k counts:
cluster_unknown_k
0      35
1      54
2     229
3      39
4      28
     ... 
91     18
92     13
93     26
94     51
95    129
Name: count, Length: 96, dtype: int64
cluster_spec counts:
cluster_spec
0      90
1     102
2     262
3     115
4      33
     ... 
91     10
92     15
93     11
94     13
95     28
Name: count, Length: 96, dtype: int64
cluster_llmedgerefine counts:
cluster_llmedgerefine
0      89
1      96
2     262
3     119
4      35
     ... 
91     10
92     14
93     11
94     14
95     28
Name: count, Length: 96, dtype: int64


,iteration,row_index,query_text,current_cluster,nearest_cluster_by_centroid,candidate_clusters,llm_selected_cluster,should_remove,assigned_cluster_after_refinement,raw_output,error
0,1,127,designed_by,77,77,"[77, 8, 16, 5, 52, 1, 2, 9]",77.0,False,77,77,None
1,1,33,based_on_novel_by,83,83,"[83, 16, 5, 1, 85, 2, 24, 77]",83.0,False,83,83,None
2,1,76,position_held_at_organization,23,23,"[23, 81, 7, 65, 2, 80, 37, 13]",23.0,False,23,23,None
3,1,56,event_location_governance,79,79,"[79, 65, 0, 3, 44, 60, 2, 12]",65.0,True,79,65,None
4,1,115,designed_by,77,77,"[77, 8, 16, 5, 52, 1, 2, 9]",77.0,False,77,77,None


In [9]:
save_path = "/workspace/KG-known0/ARR2026/Baselines/3_LLMEdgeRefine/fewrel_train_llmedgerefine_result.csv"
log_path = "/workspace/KG-known0/ARR2026/Baselines/3_LLMEdgeRefine/fewrel_train_llmedgerefine_log.csv"

df_train.to_csv(save_path, index=False)
llmedgerefine_log_df.to_csv(log_path, index=False)

print("saved:", save_path)
print("saved:", log_path)

saved: /workspace/KG-known0/ARR2026/Baselines/3_LLMEdgeRefine/fewrel_train_llmedgerefine_result.csv
saved: /workspace/KG-known0/ARR2026/Baselines/3_LLMEdgeRefine/fewrel_train_llmedgerefine_log.csv


# Test set Evaluation

In [10]:
test_df = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/FewRel/Test/FewRel_test_phrase.csv')
print(test_df.shape)
test_df.head(2)

(1600, 14)


,id,relation,relation_id,text,subject,object,subj_start,subj_end,obj_start,obj_end,subj_id,obj_id,token,gpt4_phrase
0,train_P1877_500,after a work by,P1877,"In 1998 , she starred opposite Elisabeth Shue ...",cousin bette,balzac,16,17,13,13,Q835676,Q9711,"['In', '1998', ',', 'she', 'starred', 'opposit...",work_author
1,train_P1877_396,after a work by,P1877,"He directed Disney 's "" Tom and Huck "" in 1995...",tom and huck,mark twain,5,7,15,16,Q1616185,Q7245,"['He', 'directed', 'Disney', ""'s"", '""', 'Tom',...",adaptation_author


In [11]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    adjusted_rand_score,
    v_measure_score,
    homogeneity_score,
    completeness_score,
    confusion_matrix
)
from scipy.optimize import linear_sum_assignment


def evaluate_clustering(df: pd.DataFrame, pred_col: str, gold_col: str = "relation"):

    
    non_na_df = df[~df[pred_col].isna()].copy()

    if len(non_na_df) == 0:
        raise ValueError(f"{pred_col}에 유효한 값이 없습니다.")

    y_true = non_na_df[gold_col].astype(str).values
    y_pred = non_na_df[pred_col].astype(str).values

    
    overall_purity = (
        non_na_df
        .groupby(pred_col)[gold_col]
        .value_counts()
        .groupby(level=0)
        .apply(lambda x: x.max())
        .sum()
        / len(non_na_df)
    )

    
    ari = adjusted_rand_score(y_true, y_pred)

    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v_f1 = v_measure_score(y_true, y_pred)

    
    def b3_precision_recall_f1(y_true, y_pred):

        true_groups = {}
        pred_groups = {}

        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            true_groups.setdefault(t, set()).add(i)
            pred_groups.setdefault(p, set()).add(i)

        P_sum, R_sum = 0.0, 0.0
        n = len(y_true)

        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            T = true_groups[t]
            C = pred_groups[p]
            inter = len(T & C)

            P_sum += inter / len(C) if len(C) > 0 else 0.0
            R_sum += inter / len(T) if len(T) > 0 else 0.0

        P = P_sum / n
        R = R_sum / n
        F1 = (2 * P * R / (P + R)) if (P + R) > 0 else 0.0

        return P, R, F1

    b3_p, b3_r, b3_f1 = b3_precision_recall_f1(y_true, y_pred)

    
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    hungarian_acc = cm[row_ind, col_ind].sum() / cm.sum()

    
    print(f"Purity : {overall_purity:.4f}")
    print(f"ACC    : {hungarian_acc:.4f}")
    
    
    print(f"B3-F1  : {b3_f1:.4f}")
    
    
    print(f"V-F1   : {v_f1:.4f}")
    print(f"ARI    : {ari:.4f}")

    return {
        "Purity": overall_purity,
        "ARI": ari,
        "V-hom": hom,
        "V-com": com,
        "V-F1": v_f1,
        "B3-P": b3_p,
        "B3-R": b3_r,
        "B3-F1": b3_f1,
        "ACC": hungarian_acc,
    }

In [12]:
from sklearn.metrics.pairwise import euclidean_distances
import numpy as np
import pandas as pd

if "gpt4_phrase" not in test_df.columns:
    raise ValueError("test_df에 'gpt4_phrase' 컬럼이 없습니다.")

if "relation" not in test_df.columns:
    raise ValueError("test_df에 gold label 컬럼 'relation'이 없습니다.")

if "cluster_llmedgerefine" not in df_train.columns:
    raise ValueError("df_train에 'cluster_llmedgerefine' 컬럼이 없습니다. 먼저 LLMEdgeRefine을 실행해야 합니다.")

test_texts = test_df["gpt4_phrase"].fillna("").astype(str).tolist()

print("num test instances:", len(test_texts))

test_embeddings = model.encode(
    test_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("test_embeddings shape:", test_embeddings.shape)


def build_train_cluster_centroids(train_embeddings, train_labels):
    train_labels = np.asarray(train_labels)
    unique_clusters = sorted(np.unique(train_labels))

    centroids = []
    for cluster_id in unique_clusters:
        idx = np.where(train_labels == cluster_id)[0]
        centroid = train_embeddings[idx].mean(axis=0)
        centroids.append(centroid)

    centroids = np.vstack(centroids)

    return centroids, unique_clusters


def assign_test_to_refined_clusters(test_embeddings, train_centroids, unique_clusters):
    dists = euclidean_distances(test_embeddings, train_centroids)

    nearest_local_indices = np.argmin(dists, axis=1)
    assigned_clusters = np.array(
        [unique_clusters[i] for i in nearest_local_indices],
        dtype=int
    )

    assigned_distances = np.min(dists, axis=1)

    return assigned_clusters, assigned_distances


train_labels_refined = df_train["cluster_llmedgerefine"].to_numpy()

refined_centroids, refined_cluster_ids = build_train_cluster_centroids(
    train_embeddings=embeddings,
    train_labels=train_labels_refined
)

print("num refined train clusters:", len(refined_cluster_ids))
print("refined_centroids shape:", refined_centroids.shape)

test_assigned_clusters, test_assigned_distances = assign_test_to_refined_clusters(
    test_embeddings=test_embeddings,
    train_centroids=refined_centroids,
    unique_clusters=refined_cluster_ids
)

test_df["cluster_llmedgerefine"] = test_assigned_clusters
test_df["distance_to_refined_centroid"] = test_assigned_distances

print(test_df[[
    "relation",
    "gpt4_phrase",
    "cluster_llmedgerefine",
    "distance_to_refined_centroid"
]].head(10))

metrics_llmedgerefine_test = evaluate_clustering(
    df=test_df,
    pred_col="cluster_llmedgerefine",
    gold_col="relation"
)

metrics_llmedgerefine_test

num test instances: 1600


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

test_embeddings shape: (1600, 384)
num refined train clusters: 96
refined_centroids shape: (96, 384)
          relation                        gpt4_phrase  cluster_llmedgerefine  \
0  after a work by                        work_author                     24   
1  after a work by                  adaptation_author                     25   
2  after a work by                    work_authorship                     24   
3  after a work by                    based_on_author                     83   
4  after a work by                  adaptation_author                     25   
5  after a work by                 inspiration_author                     24   
6  after a work by                        work_author                     24   
7  after a work by                  adaptation_author                     25   
8  after a work by                        inspired_by                      1   
9  after a work by  based_on_author_of_original_novel                     83   

   distance_to_ref

{'Purity': 0.518125,
 'ARI': 0.2718401111504289,
 'V-hom': 0.6862410848302664,
 'V-com': 0.6926403031151105,
 'V-F1': 0.6894258449816819,
 'B3-P': 0.41140862198258304,
 'B3-R': 0.37562499999999943,
 'B3-F1': 0.39270333392599965,
 'ACC': 0.46}

In [13]:

metrics_llmedgerefine_test = evaluate_clustering(
    df=test_df,
    pred_col="cluster_llmedgerefine",
    gold_col="relation"
)

Purity : 0.5181
ACC    : 0.4600
B3-F1  : 0.3927
V-F1   : 0.6894
ARI    : 0.2718
